# Part 2 — restore the saved T5 models on your new GPU

Use your **current L4 Colab runtime**. No manual Part 2 upload is needed. Open this notebook, connect it to the existing L4 session, then run Sections **1 → 2 → 3**, one code cell at a time. Do not use Run All. Leave the notebook kernel as Colab Python; commands explicitly use the separate Python 3.11 environment.

This notebook restores your original Drive run `20261110T032045677993Z` into a new local folder. It verifies the source-file hashes, safely extracts both model archives, installs the original Part 2 requirements, tests CUDA, reloads the saved models, and evaluates both on the 466 dev examples. It does **not** start training, create an optimizer, or use the held-out test dataset. Your local Mac homework files and the original Drive run are not edited.

Expected backups: `best.tar` from epoch 3, and `last.tar` from epoch 4. The actual epochs are read from archive metadata, not assumed. New logs, restored-model dev predictions, environment details, and `recovery_report.json` are saved under `MyDrive/NYU_NLP_HW4/part2/recovery_runs/<timestamp>/`.

Package installation and two complete dev evaluations can take several minutes. Wait for each section to finish before moving on. The final success message is **`PART 2 RECOVERY AND DEV CHECK PASSED`**. Save this notebook afterward and ask for review before continuing training.

The checkpoint archives preserve weights/config/tokenizer, but not optimizer/scheduler/RNG state. They restore the model, not the exact interrupted training process. Continuing training will need a separately documented restarted optimizer/scheduler.


## 1. Restore code, data, and both checkpoint archives

Authorize Drive if asked. This only copies into a new local folder and creates a separate recovery-log folder. Run once.


In [1]:
RECOVERY_TOOLS_SOURCE = "\"\"\"Filesystem-only recovery helpers. Never overwrite the original Drive run.\"\"\"\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport tarfile\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\"rb\") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef child_path(root, relative):\n    root = Path(root).resolve()\n    relative = PurePosixPath(relative)\n    if relative.is_absolute() or \"..\" in relative.parts or \"\\\\\" in str(relative):\n        raise ValueError(f\"Unsafe relative path: {relative}\")\n    result = (root / str(relative)).resolve()\n    if result != root and root not in result.parents:\n        raise ValueError(f\"Path escapes recovery root: {relative}\")\n    return result\n\n\ndef restore_source(original_run, work):\n    original_run, work = Path(original_run), Path(work)\n    source = original_run / \"source\"\n    manifest = json.loads((original_run / \"source_sha256.json\").read_text())\n    if not manifest:\n        raise ValueError(\"Empty source manifest.\")\n    for name, expected in manifest.items():\n        src = child_path(source, name)\n        dst = child_path(work, name)\n        if not src.is_file() or sha256_file(src) != expected:\n            raise ValueError(f\"Original source hash mismatch: {name}\")\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        with dst.open(\"xb\") as handle, src.open(\"rb\") as incoming:\n            shutil.copyfileobj(incoming, handle)\n        if sha256_file(dst) != expected:\n            raise ValueError(f\"Restored source hash mismatch: {name}\")\n    # The original backup deliberately stored this pickle at the source root.\n    records = work / \"records\"\n    records.mkdir(exist_ok=True)\n    with (records / \"ground_truth_dev.pkl\").open(\"xb\") as handle:\n        handle.write((work / \"ground_truth_dev.pkl\").read_bytes())\n    for name in [\"load_data.py\", \"t5_utils.py\", \"train_t5.py\", \"utils.py\", \"requirements.txt\",\n                 \"data/train.nl\", \"data/train.sql\", \"data/dev.nl\", \"data/dev.sql\",\n                 \"data/flight_database.db\", \"records/ground_truth_dev.pkl\"]:\n        if not (work / name).is_file():\n            raise FileNotFoundError(f\"Incomplete restored source: {name}\")\n    return {\"verified_files\": len(manifest), \"source_manifest\": str(original_run / \"source_sha256.json\")}\n\n\ndef restore_checkpoint(archive_path, local_archive, checkpoint_root, name):\n    if name not in {\"best\", \"last\"}:\n        raise ValueError(name)\n    archive_path, local_archive = Path(archive_path), Path(local_archive)\n    local_archive.parent.mkdir(parents=True, exist_ok=True)\n    with local_archive.open(\"xb\") as target, archive_path.open(\"rb\") as source:\n        shutil.copyfileobj(source, target)\n    if local_archive.stat().st_size != archive_path.stat().st_size:\n        raise ValueError(\"Checkpoint archive copy is incomplete.\")\n    checkpoint_root = Path(checkpoint_root)\n    if (checkpoint_root / name).exists():\n        raise FileExistsError(f\"Refusing to overwrite restored checkpoint: {name}\")\n    checkpoint_root.mkdir(parents=True, exist_ok=True)\n    with tarfile.open(local_archive, \"r:\") as archive:\n        members = archive.getmembers()\n        seen = set()\n        for member in members:\n            path = PurePosixPath(member.name)\n            child_path(checkpoint_root, member.name)\n            if not path.parts or path.parts[0] != name:\n                raise ValueError(f\"Unexpected archive root: {member.name}\")\n            if not (member.isdir() or member.isfile()) or member.name in seen:\n                raise ValueError(f\"Unsafe/duplicate tar member: {member.name}\")\n            seen.add(member.name)\n        required = {f\"{name}/{p}\" for p in\n                    [\"config.json\", \"model.safetensors\", \"tokenizer.json\", \"checkpoint_meta.json\"]}\n        if not required.issubset(seen):\n            raise ValueError(\"Archive lacks required model/tokenizer/metadata files.\")\n        for member in members:\n            target = child_path(checkpoint_root, member.name)\n            if member.isdir():\n                target.mkdir(parents=True, exist_ok=True)\n                continue\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with archive.extractfile(member) as source, target.open(\"xb\") as output:\n                shutil.copyfileobj(source, output)\n            if target.stat().st_size != member.size:\n                raise ValueError(f\"Incomplete extracted file: {member.name}\")\n    checkpoint = checkpoint_root / name\n    metadata = json.loads((checkpoint / \"checkpoint_meta.json\").read_text())\n    if not isinstance(metadata.get(\"epoch\"), int) or metadata[\"epoch\"] < 1:\n        raise ValueError(\"Checkpoint epoch metadata is invalid.\")\n    return {\"name\": name, \"epoch\": metadata[\"epoch\"], \"metadata\": metadata,\n            \"original_archive\": str(archive_path), \"local_directory\": str(checkpoint),\n            \"archive_bytes\": local_archive.stat().st_size,\n            \"local_archive_sha256\": sha256_file(local_archive)}\n"

# The notebook embeds RECOVERY_TOOLS_SOURCE before this preparation code.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import importlib.util
import json
import shutil
import subprocess
import sys
import traceback

assert sys.platform == "linux" and Path("/content").exists(), "Select Colab, not Mac Python."
drive.mount("/content/drive")
ORIGINAL_RUN = Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z")
for name in ["source_sha256.json", "configuration.json", "checkpoints/best.tar", "checkpoints/last.tar"]:
    assert (ORIGINAL_RUN / name).is_file(), f"Missing original backup: {name}. Stop and ask for help."
assert shutil.disk_usage("/content").free > 6 * 1024**3, "Need at least 6 GiB local space for environment/models."
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RECOVERY_WORK = Path("/content") / f"hw4_t5_recovery_{stamp}"
RECOVERY_BACKUP = ORIGINAL_RUN.parent.parent / "recovery_runs" / stamp
RECOVERY_WORK.mkdir(exist_ok=False)
RECOVERY_BACKUP.mkdir(parents=True, exist_ok=False)
ENV2 = Path("/content/hw4-part2-env")
PYTHON2 = str(ENV2 / "bin/python")

def recovery_status(state, stage, **extra):
    path = RECOVERY_BACKUP / "status.json"
    status = json.loads(path.read_text()) if path.is_file() else {}
    status.update(state=state, stage=stage, updated_utc=datetime.now(timezone.utc).isoformat(),
                  original_run=str(ORIGINAL_RUN), recovery_work=str(RECOVERY_WORK),
                  recovery_backup=str(RECOVERY_BACKUP), training_updates=0, held_out_test_used=False,
                  **extra)
    path.write_text(json.dumps(status, indent=2))

def run_recovery_command(command, log_name):
    print("\nRunning:", " ".join(map(str, command)), flush=True)
    with (RECOVERY_BACKUP / log_name).open("a", buffering=1) as log:
        process = subprocess.Popen(list(map(str, command)), cwd=RECOVERY_WORK,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait(timeout=15)
            raise
        finally:
            process.stdout.close()
        if code:
            raise RuntimeError(f"Command exited with code {code}. See {log_name} in the recovery folder.")

recovery_status("RUNNING", "restoring_source_and_checkpoints")
try:
    tools_file = RECOVERY_WORK / "recovery_tools.py"
    tools_file.write_text(RECOVERY_TOOLS_SOURCE)
    shutil.copy2(tools_file, RECOVERY_BACKUP / tools_file.name)
    spec = importlib.util.spec_from_file_location("hw4_recovery_tools", tools_file)
    recovery_tools = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(recovery_tools)
    source_info = recovery_tools.restore_source(ORIGINAL_RUN, RECOVERY_WORK)
    print("Restored code/data; SHA-256 verified files:", source_info["verified_files"], flush=True)
    settings = json.loads((ORIGINAL_RUN / "configuration.json").read_text())
    assert settings["finetune"] is True
    experiment = settings["experiment_name"]
    assert experiment and Path(experiment).name == experiment and experiment not in {".", ".."}
    root = RECOVERY_WORK / "checkpoints/ft_experiments" / experiment
    checkpoints = {}
    for name in ["best", "last"]:
        print(f"Copying and extracting {name}.tar from Drive...", flush=True)
        checkpoints[name] = recovery_tools.restore_checkpoint(
            ORIGINAL_RUN / "checkpoints" / f"{name}.tar",
            RECOVERY_WORK / "archives" / f"{name}.tar", root, name,
        )
        print(f"RESTORED {name}: epoch {checkpoints[name]['epoch']}", flush=True)
    context = {"original_run": str(ORIGINAL_RUN), "work_directory": str(RECOVERY_WORK),
               "recovery_backup": str(RECOVERY_BACKUP), "configuration": settings,
               "source_verification": source_info, "checkpoints": checkpoints,
               "training_updates": 0, "held_out_test_used": False}
    CONTEXT_PATH = RECOVERY_BACKUP / "recovery_context.json"
    CONTEXT_PATH.write_text(json.dumps(context, indent=2))
    recovery_status("RESTORED_NOT_YET_MODEL_CHECKED", "source_and_archives_restored")
except BaseException:
    recovery_status("FAILED", "source_or_archive_restore_failed", error=traceback.format_exc())
    raise
print("\nSOURCE AND CHECKPOINT ARCHIVES RESTORED. No training has started.")
print("Local working folder:", RECOVERY_WORK)
print("NEW RECOVERY LOG/REPORT FOLDER:", RECOVERY_BACKUP)
print("Original Drive backup has NOT been changed. Next run Section 2 to install the environment.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restored code/data; SHA-256 verified files: 14
Copying and extracting best.tar from Drive...
RESTORED best: epoch 3
Copying and extracting last.tar from Drive...
RESTORED last: epoch 4

SOURCE AND CHECKPOINT ARCHIVES RESTORED. No training has started.
Local working folder: /content/hw4_t5_recovery_20261110T132529577782Z
NEW RECOVERY LOG/REPORT FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/recovery_runs/20261110T132529577782Z
Original Drive backup has NOT been changed. Next run Section 2 to install the environment.


## 2. Rebuild the Part 2 environment and verify the GPU

Install the original pinned packages in Python 3.11. This does not train. Wait for `PART 2 RECOVERY ENVIRONMENT READY`.


In [2]:
# Rebuild the same Python 3.11 / Part 2 package versions, separate from kernel Python.
assert "RECOVERY_WORK" in globals() and (RECOVERY_WORK / "requirements.txt").is_file(), "Run Section 1 first."
recovery_status("RUNNING", "installing_part2_environment")
try:
    run_recovery_command([sys.executable, "-m", "pip", "install", "uv"], "environment_install.log")
    uv = [sys.executable, "-m", "uv"]
    run_recovery_command(uv + ["python", "install", "3.11"], "environment_install.log")
    if not Path(PYTHON2).is_file():
        run_recovery_command(uv + ["venv", "--python", "3.11", "--seed", str(ENV2)], "environment_install.log")
    run_recovery_command(uv + ["pip", "install", "--python", PYTHON2,
                              "-r", str(RECOVERY_WORK / "requirements.txt")], "environment_install.log")
    run_recovery_command(uv + ["pip", "check", "--python", PYTHON2], "environment_install.log")
    ENVIRONMENT_CHECK = '''import importlib.metadata, json, sys
from pathlib import Path
from packaging.requirements import Requirement
import torch
assert sys.version_info[:2] == (3, 11)
versions = {}
for line in Path(sys.argv[1]).read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"): continue
    requirement = Requirement(line)
    actual = importlib.metadata.version(requirement.name)
    assert requirement.specifier.contains(actual, prereleases=True), (line, actual)
    versions[requirement.name] = actual
assert torch.cuda.is_available(), "Select your Colab GPU runtime."
x = torch.ones((16, 16), device="cuda")
assert (x @ x).sum().item() == 4096
report = {"python": sys.version, "versions": versions, "gpu": torch.cuda.get_device_name(0)}
Path(sys.argv[2]).write_text(json.dumps(report, indent=2))
print("RECOVERY GPU:", report["gpu"])
print("PART 2 RECOVERY ENVIRONMENT READY")
'''
    run_recovery_command([PYTHON2, "-u", "-c", ENVIRONMENT_CHECK,
                          str(RECOVERY_WORK / "requirements.txt"),
                          str(RECOVERY_BACKUP / "environment.json")], "environment_install.log")
    freeze = subprocess.run([PYTHON2, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
    (RECOVERY_BACKUP / "pip_freeze.txt").write_text(freeze.stdout)
    recovery_status("ENVIRONMENT_READY", "environment_ready_no_training")
except BaseException:
    recovery_status("FAILED", "environment_setup_failed", error=traceback.format_exc())
    raise
print("Next run Section 3 to reload and evaluate the two saved models. No training has started.")



Running: /usr/bin/python3 -m pip install uv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 171.6 MB/s  0:00:00

Running: /usr/bin/python3 -m uv python install 3.11
Installed Python 3.11.18 in 872ms
 + cpython-3.11.18-linux-x86_64-gnu (python3.11)

Running: /usr/bin/python3 -m uv venv --python 3.11 --seed /content/hw4-part2-env
Using CPython 3.11.18
Creating virtual environment with seed packages at: /content/hw4-part2-env
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source /content/hw4-part2-env/bin/activate

Running: /usr/bin/python3 -m uv pip install --python /content/hw4-part2-env/bin/python -r /content/hw4_t5_recovery_20261110T132529577782Z/requirements.txt
Using Python 3.11.18 environment at: /content/hw4-part2-env
Resolved 76 packages in 1.31s
Prepared 74 packages in 20.70s
Installed 74 packages in 257ms
 + accelerate==0.29.3
 + appdirs==1.4.4
 + attrs==26.1.0
 + bitsandbytes==0.43.1
 + certifi==2026.10.23
 + charset-normalizer=

## 3. Reload and evaluate the restored checkpoints — no training

Run after Sections 1 and 2 finish. This checks both models and saved tokenizers, then evaluates the best and latest models on dev only. New SQL/pickle outputs and logs go directly to the separate recovery folder on Drive. No checkpoint in the original run is overwritten.


In [3]:
RECOVERY_EVALUATE_SOURCE = "\"\"\"Reload and evaluate saved best/latest checkpoints. NO training or held-out test.\"\"\"\n\nimport gc\nimport json\nimport math\nfrom pathlib import Path\nimport pickle\nimport sys\nimport time\nimport traceback\nfrom datetime import datetime, timezone\nfrom types import SimpleNamespace\n\nimport torch\nfrom transformers import T5TokenizerFast\n\n\ndef main(context_path):\n    context = json.loads(Path(context_path).read_text())\n    work = Path(context[\"work_directory\"]).resolve()\n    backup = Path(context[\"recovery_backup\"]).resolve()\n    assert Path.cwd().resolve() == work, \"Run with cwd set to the restored working folder.\"\n    sys.path.insert(0, str(work))\n    from load_data import get_dataloader, preprocess_sql\n    from t5_utils import load_model_from_checkpoint\n    from train_t5 import eval_epoch\n    from utils import set_random_seeds\n\n    args = SimpleNamespace(**context[\"configuration\"])\n    assert args.finetune\n    assert torch.cuda.is_available(), \"Select your L4 (or another CUDA GPU).\"\n    set_random_seeds(args.seed)\n    started = time.monotonic()\n    report = {\"original_run\": context[\"original_run\"], \"recovery_backup\": str(backup),\n              \"work_directory\": str(work), \"gpu\": torch.cuda.get_device_name(0),\n              \"training_updates\": 0, \"held_out_test_used\": False,\n              \"checkpoints\": {}}\n\n    def update(state, stage, **extra):\n        path = backup / \"status.json\"\n        status = json.loads(path.read_text()) if path.is_file() else {}\n        status.update(state=state, stage=stage, updated_utc=datetime.now(timezone.utc).isoformat(),\n                      training_updates=0, held_out_test_used=False, **extra)\n        path.write_text(json.dumps(status, indent=2))\n\n    try:\n        update(\"RUNNING\", \"loading_dev_data\")\n        # Only the tokenizer may need a small first-time Hugging Face download.\n        # Model weights below load from local restored checkpoints only.\n        dev_loader = get_dataloader(args.test_batch_size, \"dev\")\n        assert len(dev_loader.dataset) == 466\n        raw_nl = (work / \"data/dev.nl\").read_text().splitlines()\n        raw_sql = (work / \"data/dev.sql\").read_text().splitlines()\n        assert len(raw_nl) == len(raw_sql) == 466\n        with (work / \"records/ground_truth_dev.pkl\").open(\"rb\") as handle:\n            gold_records, gold_errors = pickle.load(handle)\n        assert len(gold_records) == len(gold_errors) == 466\n\n        for name in [\"best\", \"last\"]:\n            info = context[\"checkpoints\"][name]\n            checkpoint = Path(info[\"local_directory\"])\n            epoch = info[\"epoch\"]\n            update(\"RUNNING\", f\"loading_{name}_checkpoint\", checkpoint_epoch=epoch)\n            print(f\"\\nLOADING RESTORED {name.upper()} CHECKPOINT: epoch {epoch}\", flush=True)\n            tokenizer = T5TokenizerFast.from_pretrained(str(checkpoint), local_files_only=True)\n            assert tokenizer.get_vocab() == dev_loader.dataset.tokenizer.get_vocab()\n            # Ensure saved and freshly cached tokenizer behavior agrees on all dev data.\n            assert tokenizer(raw_nl, padding=False, truncation=False)[\"input_ids\"] == dev_loader.dataset.input_ids\n            assert tokenizer([preprocess_sql(q) for q in raw_sql], padding=False, truncation=False)[\"input_ids\"] == dev_loader.dataset.target_ids\n            model = load_model_from_checkpoint(args, best=(name == \"best\"))\n            assert next(model.parameters()).device.type == \"cuda\"\n            assert model.config.model_type == \"t5\" and model.config.d_model == 512\n            assert model.config.decoder_start_token_id == model.config.pad_token_id == 0\n            assert model.config.eos_token_id == 1\n            parameters = sum(p.numel() for p in model.parameters())\n            assert parameters == 60_506_624\n            print(f\"Loaded {parameters:,} parameters on {report['gpu']}; tokenizer verified.\", flush=True)\n            signature = model.shared.weight[:16, :16].detach().clone()\n            update(\"RUNNING\", f\"evaluating_{name}_on_dev\", checkpoint_epoch=epoch)\n            sql_path = backup / f\"{name}_epoch_{epoch}_dev.sql\"\n            records_path = backup / f\"{name}_epoch_{epoch}_dev.pkl\"\n            metrics = eval_epoch(\n                args, model, dev_loader, str(work / \"data/dev.sql\"), str(sql_path),\n                str(work / \"records/ground_truth_dev.pkl\"), str(records_path),\n            )\n            loss, f1, record_em, sql_em, error_rate = map(float, metrics)\n            assert all(math.isfinite(v) for v in [loss, f1, record_em, sql_em, error_rate])\n            assert all(0 <= v <= 1 for v in [f1, record_em, sql_em, error_rate])\n            assert len(sql_path.read_text().splitlines()) == 466\n            with records_path.open(\"rb\") as handle:\n                records, errors = pickle.load(handle)\n            assert len(records) == len(errors) == 466\n            assert not model.training and all(p.grad is None for p in model.parameters())\n            assert torch.equal(signature, model.shared.weight[:16, :16].detach())\n            result = {\"epoch\": epoch, \"parameters\": parameters, \"dev_examples\": 466,\n                      \"dev_loss\": loss, \"record_f1\": f1, \"record_em\": record_em,\n                      \"sql_em\": sql_em, \"sql_error_rate\": error_rate,\n                      \"sql_path\": str(sql_path), \"records_path\": str(records_path)}\n            previous = info[\"metadata\"].get(\"record_f1\")\n            if previous is not None:\n                result[\"original_saved_record_f1\"] = previous\n                result[\"absolute_f1_difference_after_gpu_change\"] = abs(f1 - previous)\n                if abs(f1 - previous) > 1e-6:\n                    print(\"NOTE: re-evaluated F1 differs from the old GPU result; inspect before continuing.\", flush=True)\n            report[\"checkpoints\"][name] = result\n            (backup / \"recovery_results_partial.json\").write_text(json.dumps(report, indent=2))\n            print(f\"RECOVERED {name.upper()} EPOCH {epoch}: dev loss={loss:.4f}, \"\n                  f\"Record F1={100*f1:.2f}%, Record EM={100*record_em:.2f}%, \"\n                  f\"SQL EM={100*sql_em:.2f}%, SQL errors={100*error_rate:.2f}%\", flush=True)\n            del model, tokenizer, signature\n            gc.collect()\n            torch.cuda.empty_cache()\n\n        best_candidate = max(report[\"checkpoints\"], key=lambda name: report[\"checkpoints\"][name][\"record_f1\"])\n        report.update(state=\"COMPLETED\", seconds=time.monotonic() - started,\n                      highest_dev_f1_checkpoint=best_candidate,\n                      highest_dev_f1_epoch=report[\"checkpoints\"][best_candidate][\"epoch\"],\n                      latest_training_epoch=context[\"checkpoints\"][\"last\"][\"epoch\"],\n                      optimizer_state_available=False, exact_training_resume_available=False)\n        (backup / \"recovery_report.json\").write_text(json.dumps(report, indent=2))\n        update(\"COMPLETED\", \"both_checkpoints_reloaded_and_dev_evaluated\")\n        print(\"\\nPART 2 RECOVERY AND DEV CHECK PASSED\", flush=True)\n        print(json.dumps(report, indent=2), flush=True)\n        print(\"No training updates. No held-out test inputs used. Original Drive run unchanged.\")\n        print(\"Save this notebook and ask for review before continuing training.\")\n    except BaseException:\n        update(\"FAILED\", \"model_reload_or_dev_check_failed\", error=traceback.format_exc())\n        raise\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1])\n"

# The notebook embeds RECOVERY_EVALUATE_SOURCE before this code.
assert "RECOVERY_WORK" in globals() and Path(PYTHON2).is_file(), "Run Sections 1 and 2 first."
assert (RECOVERY_BACKUP / "environment.json").is_file(), "Complete the Section 2 GPU/environment check first."
state = json.loads((RECOVERY_BACKUP / "status.json").read_text())["state"]
assert state != "COMPLETED", "This recovery check already completed; inspect its report instead of rerunning."
assert state in {"ENVIRONMENT_READY", "FAILED"}, "Complete environment setup first."
script = RECOVERY_WORK / "check_recovered_models.py"
script.write_text(RECOVERY_EVALUATE_SOURCE)
shutil.copy2(script, RECOVERY_BACKUP / script.name)
try:
    run_recovery_command([PYTHON2, "-u", str(script), str(CONTEXT_PATH)], "recovery_evaluation.log")
except BaseException:
    recovery_status("FAILED", "recovery_evaluation_failed_or_interrupted", error=traceback.format_exc())
    print("Existing recovery logs/results remain at:", RECOVERY_BACKUP)
    raise
else:
    assert json.loads((RECOVERY_BACKUP / "status.json").read_text())["state"] == "COMPLETED"
    print("\nRECOVERY REPORT AND LOGS:", RECOVERY_BACKUP)
    print("Save this notebook and ask for review. No training has been started.")



Running: /content/hw4-part2-env/bin/python -u /content/hw4_t5_recovery_20261110T132529577782Z/check_recovered_models.py /content/drive/MyDrive/NYU_NLP_HW4/part2/recovery_runs/20261110T132529577782Z/recovery_context.json
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

LOADING RESTORED BEST CHECKPOINT: epoch 3
Loaded 60,506,624 parameters on NVIDIA L4; tokenizer verified.

Dev loss + generation: 100%|██████████| 59/59 [03:15<00:00,  3.31s/it]

0it [00:00, ?it/s]
17it [00:00, 168.15it/s]
61it [00:00, 325.55it/s]
98it [00:00, 344.50it/s]
138it [00:00, 365.10it/s]
191it [00:00, 421.39it/s]
234it [00:00, 408.22it/s]
275it [00:00, 369.52it/s]
3

KeyboardInterrupt: 

## 4. Inspect the recovery report — read-only

Safe after reconnecting. Do not rerun restore/evaluation merely to see the saved results.


In [4]:
# Read-only: can run later without reinstalling packages or restoring again.
from pathlib import Path
from google.colab import drive
import json

drive.mount("/content/drive")
if "RECOVERY_BACKUP" not in globals():
    folders = sorted(Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/recovery_runs").glob("*"))
    assert folders, "No recovery folders found; first run Section 1."
    RECOVERY_BACKUP = folders[-1]
print("RECOVERY FOLDER:", RECOVERY_BACKUP)
for name in ["status.json", "environment.json", "recovery_report.json", "recovery_results_partial.json"]:
    file = RECOVERY_BACKUP / name
    if file.is_file():
        if name == "recovery_results_partial.json" and (RECOVERY_BACKUP / "recovery_report.json").is_file():
            continue
        print("\n" + name + ":\n" + file.read_text())
log = RECOVERY_BACKUP / "recovery_evaluation.log"
if log.is_file():
    print("\nLatest log lines:\n" + "\n".join(log.read_text(errors="replace").splitlines()[-15:]))
print("Inspection does not start or resume training.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RECOVERY FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/recovery_runs/20261110T132529577782Z

status.json:
{
  "state": "FAILED",
  "stage": "recovery_evaluation_failed_or_interrupted",
  "updated_utc": "2026-11-10T13:53:45.958117+00:00",
  "original_run": "/content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z",
  "recovery_work": "/content/hw4_t5_recovery_20261110T132529577782Z",
  "recovery_backup": "/content/drive/MyDrive/NYU_NLP_HW4/part2/recovery_runs/20261110T132529577782Z",
  "training_updates": 0,
  "held_out_test_used": false,
  "checkpoint_epoch": 4,
  "error": "Traceback (most recent call last):\n  File \"/tmp/ipykernel_7803/2143298794.py\", line 13, in <cell line: 0>\n    run_recovery_command([PYTHON2, \"-u\", str(script), str(CONTEXT_PATH)], \"recovery_evaluation.log\")\n    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^